# Exercise 1.2 - Directional Car Counting (Downtown → City Centre)

This notebook counts vehicles that move from **downtown** to **city centre** using:
- frame differencing,
- background subtraction,
- ROI restriction,
- contour filtering,
- centroid tracking with one-time counting per vehicle.

The notebook processes both required videos:
- `Traffic_Laramie_1.mp4`
- `Traffic_Laramie_2.mp4`

In [6]:
%pip install -q opencv-python numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from pathlib import Path
import cv2
import numpy as np

VIDEO_FILES = [Path("Traffic_Laramie_1.mp4"), Path("Traffic_Laramie_2.mp4")]
DISPLAY_VIDEO = False  # Set True to show live OpenCV windows.

for video_path in VIDEO_FILES:
    assert video_path.exists(), f"Missing video: {video_path.resolve()}"

print("Input videos found:")
for video_path in VIDEO_FILES:
    print("-", video_path.name)

Input videos found:
- Traffic_Laramie_1.mp4
- Traffic_Laramie_2.mp4


In [8]:
def main_street_roi_mask(frame_shape):
    h, w = frame_shape[:2]
    pts = np.array([
        [int(0.02 * w), int(0.53 * h)],
        [int(0.78 * w), int(0.53 * h)],
        [int(0.98 * w), int(0.93 * h)],
        [int(0.01 * w), int(0.93 * h)],
    ], dtype=np.int32)
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(mask, [pts], 255)
    return mask, pts


def centroid(box):
    x, y, bw, bh = box
    return (x + bw // 2, y + bh // 2)


def filter_vehicle_detections(binary_mask):
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    detections = []

    for contour in contours:
        area = cv2.contourArea(contour)
        if area < 850:
            continue

        x, y, w, h = cv2.boundingRect(contour)
        if w < 24 or h < 20:
            continue
        if h > 2.2 * w:
            continue

        bbox_area = w * h
        if bbox_area < 1400:
            continue

        # Reject partial blobs: full vehicles occupy a larger fraction of their box.
        extent = area / float(bbox_area)
        if extent < 0.35:
            continue

        # Exclude objects clipped by frame borders, which are usually partial vehicles.
        mask_h, mask_w = binary_mask.shape[:2]
        if x <= 0 or y <= 0 or (x + w) >= (mask_w - 1) or (y + h) >= (mask_h - 1):
            continue

        detections.append((x, y, w, h))

    return detections


def match_tracks(tracks, detections, next_id, max_distance=55, max_missed=12):
    updated_tracks = {}
    used_detection_indices = set()
    det_centroids = [centroid(d) for d in detections]

    for track_id, track_data in tracks.items():
        tx, ty = track_data["centroid"]
        best_idx = None
        best_dist = float("inf")

        for i, (dx, dy) in enumerate(det_centroids):
            if i in used_detection_indices:
                continue
            dist = ((tx - dx) ** 2 + (ty - dy) ** 2) ** 0.5
            if dist < best_dist:
                best_dist = dist
                best_idx = i

        if best_idx is not None and best_dist <= max_distance:
            used_detection_indices.add(best_idx)
            updated_tracks[track_id] = {
                "box": detections[best_idx],
                "centroid": det_centroids[best_idx],
                "previous_centroid": track_data["centroid"],
                "missed": 0,
                "counted": track_data["counted"],
            }
        else:
            missed = track_data["missed"] + 1
            if missed <= max_missed:
                updated_tracks[track_id] = {
                    "box": track_data["box"],
                    "centroid": track_data["centroid"],
                    "previous_centroid": track_data["previous_centroid"],
                    "missed": missed,
                    "counted": track_data["counted"],
                }

    for i, det in enumerate(detections):
        if i in used_detection_indices:
            continue

        c = det_centroids[i]
        updated_tracks[next_id] = {
            "box": det,
            "centroid": c,
            "previous_centroid": c,
            "missed": 0,
            "counted": False,
        }
        next_id += 1

    return updated_tracks, next_id


def has_crossed_downtown_to_citycentre(prev_centroid, curr_centroid, line_x):
    prev_x, _ = prev_centroid
    curr_x, _ = curr_centroid
    return prev_x > line_x and curr_x <= line_x


def process_video(video_path, output_path, display_video=True):
    cap = cv2.VideoCapture(str(video_path))
    assert cap.isOpened(), f"Could not open {video_path}"

    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(str(output_path), fourcc, fps, (frame_w, frame_h))

    roi_mask, roi_polygon = main_street_roi_mask((frame_h, frame_w, 3))
    line_x = int(0.60 * frame_w)
    line_y1 = int(0.56 * frame_h)
    line_y2 = int(0.94 * frame_h)

    background_subtractor = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=28, detectShadows=False)

    tracks = {}
    next_track_id = 1
    count_downtown_to_citycentre = 0
    prev_gray = None
    frame_count = 0

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    window_name = f"{video_path.name} - Downtown to City Centre Count (press q to stop)"
    show_enabled = bool(display_video)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        blur = cv2.GaussianBlur(gray, (5, 5), 0)

        fg_mask = background_subtractor.apply(blur)
        _, fg_mask = cv2.threshold(fg_mask, 200, 255, cv2.THRESH_BINARY)

        if prev_gray is None:
            diff_mask = np.zeros_like(fg_mask)
        else:
            diff = cv2.absdiff(blur, prev_gray)
            _, diff_mask = cv2.threshold(diff, 25, 255, cv2.THRESH_BINARY)

        motion_mask = cv2.bitwise_or(fg_mask, diff_mask)
        motion_mask = cv2.bitwise_and(motion_mask, roi_mask)
        motion_mask = cv2.morphologyEx(motion_mask, cv2.MORPH_OPEN, kernel, iterations=1)
        motion_mask = cv2.morphologyEx(motion_mask, cv2.MORPH_CLOSE, kernel, iterations=2)

        detections = filter_vehicle_detections(motion_mask)
        tracks, next_track_id = match_tracks(tracks, detections, next_track_id)

        vis = frame.copy()
        cv2.polylines(vis, [roi_polygon], isClosed=True, color=(0, 0, 255), thickness=2)
        cv2.line(vis, (line_x, line_y1), (line_x, line_y2), (0, 255, 255), 2)
        cv2.putText(vis, "Count line", (line_x - 85, line_y1 - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

        for track_id, track_data in tracks.items():
            x, y, w, h = track_data["box"]
            prev_c = track_data["previous_centroid"]
            curr_c = track_data["centroid"]

            if not track_data["counted"] and has_crossed_downtown_to_citycentre(prev_c, curr_c, line_x):
                track_data["counted"] = True
                count_downtown_to_citycentre += 1

            cx, cy = curr_c
            cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv2.circle(vis, (cx, cy), 3, (255, 0, 0), -1)
            cv2.putText(vis, f"ID {track_id}", (x, max(16, y - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        cv2.putText(vis, f"Downtown -> City centre: {count_downtown_to_citycentre}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
        cv2.putText(vis, f"Frame: {frame_count}", (10, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

        out.write(vis)

        if show_enabled:
            try:
                cv2.imshow(window_name, vis)
                if (cv2.waitKey(1) & 0xFF) == ord("q"):
                    break
            except cv2.error:
                show_enabled = False
                print("OpenCV GUI display is unavailable in this environment. Continuing without live window.")

        prev_gray = blur

    cap.release()
    out.release()
    try:
        cv2.destroyAllWindows()
    except cv2.error:
        pass

    duration_minutes = frame_count / fps / 60 if fps > 0 else 0
    cars_per_minute = (count_downtown_to_citycentre / duration_minutes) if duration_minutes > 0 else 0

    return {
        "video": video_path.name,
        "total_cars": int(count_downtown_to_citycentre),
        "cars_per_min": float(cars_per_minute),
        "frames_processed": int(frame_count),
        "fps": float(fps),
        "output": output_path.name,
    }

In [9]:
results = []

for video_path in VIDEO_FILES:
    output_file = Path(f"EX1_2_counted_{video_path.stem}.mp4")
    print(f"\nProcessing {video_path.name} ...")
    stats = process_video(video_path, output_file, display_video=DISPLAY_VIDEO)
    results.append(stats)
    print(f"Done: {stats['video']} | Total cars: {stats['total_cars']} | Cars/min: {stats['cars_per_min']:.2f}")


Processing Traffic_Laramie_1.mp4 ...
Done: Traffic_Laramie_1.mp4 | Total cars: 8 | Cars/min: 2.70

Processing Traffic_Laramie_2.mp4 ...
Done: Traffic_Laramie_2.mp4 | Total cars: 4 | Cars/min: 2.27


In [10]:
print("\nTask 2 results table")
print(f"{'Video':<22} {'Total number of cars':<22} {'Cars per minute':<16}")
print("-" * 62)
for row in results:
    print(f"{row['video']:<22} {row['total_cars']:<22} {row['cars_per_min']:<16.2f}")


Task 2 results table
Video                  Total number of cars   Cars per minute 
--------------------------------------------------------------
Traffic_Laramie_1.mp4  8                      2.70            
Traffic_Laramie_2.mp4  4                      2.27            
